In [142]:
def compute_exp_F(R, v_list):
    """
    Computes the exponential map for a local algebra R with maximal ideal m.
    
    Inputs:
    - R: A finite-dimensional local algebra over k.
    - v_list: A list of n basis elements for the maximal ideal m.
    
    Outputs:
    - f: A tuple of polynomials (f_1, ..., f_n)
    - F: A list representing the homogenization [x_0^r, F_1, ..., F_n]
    """
    k = R.base_ring()
    n = len(v_list)
    
    # 1. Define the full basis for R (dimension n+1) starting with 1
    basis = [R(1)] + list(v_list)
    
    # 2. Collect all elements we will need to decompose
    # This guarantees we identify every monomial that will appear in normal forms.
    products = [v * b for v in v_list for b in basis]
    all_elements = basis + products
    
    monomials_set = set()
    for elem in all_elements:
        poly = elem.lift() # Lift from quotient ring to the cover polynomial ring
        for mon in poly.monomials():
            monomials_set.add(mon)
    monomials = list(monomials_set)
    
    # 3. Build the basis matrix
    # Each row is a basis element represented as a vector over the extracted monomials.
    basis_matrix = matrix(k, len(basis), len(monomials))
    for i, b in enumerate(basis):
        poly = b.lift()
        basis_matrix[i, :] = vector(k, [poly.monomial_coefficient(m) for m in monomials])
        
    def get_coords(elem):
        """Returns the coordinate row vector of elem in our defined basis."""
        poly = elem.lift()
        vec = vector(k, [poly.monomial_coefficient(m) for m in monomials])
        # Solve c * basis_matrix = vec to find the coordinates c
        return basis_matrix.solve_left(vec)

    # 4. Construct multiplication matrices M_i for each v_i
    M_list = []
    for v in v_list:
        M = matrix(k, n+1, n+1)
        for j, b in enumerate(basis):
            M[j] = get_coords(v * b)
        M_list.append(M)

    # 5. Set up the polynomial ring for the parameters
    x_names = ['x%d' % (i+1) for i in range(n)]
    PR = PolynomialRing(k, x_names)
    xs = PR.gens()

    # 6. Construct the element W = x_1 M_1 + ... + x_n M_n
    W = matrix(PR, n+1, n+1)
    for i in range(n):
        W += xs[i] * M_list[i].change_ring(PR)

    # 7. Compute E = exp(W) = sum_{j=0}^n W^j / j!
    E = matrix(PR, n+1, n+1)
    term = identity_matrix(PR, n+1)
    E += term
    for j in range(1, n+2):
        term = (term * W) / j
        if term.is_zero():
            break
        E += term

    # 8. Extract f_1, ..., f_n
    coords = E[0]
    f = tuple(coords[1:]) # Drop the 1 at index 0

    # 9. Homogenization
    r = max([fi.degree() if not fi.is_zero() else 0 for fi in f]) if f else 0
    
    x_hom_names = ['x0'] + x_names
    PR_hom = PolynomialRing(k, x_hom_names)
    x0 = PR_hom.gen(0)
    xs_hom = PR_hom.gens()[1:]
    
    phi = PR.hom(xs_hom, PR_hom)

    F = [x0**r]
    for fi in f:
        fi_hom = phi(fi)
        if fi_hom.is_zero():
            F.append(PR_hom.zero())
        else:
            deg_fi = fi_hom.degree()
            Fi = fi_hom.homogenize(x0) * (x0**(r - deg_fi))
            F.append(Fi)

    return f, F

def compute_base_locus(F_list):
    R = F_list[0].parent()
    P = ProjectiveSpace(R)
    try:
        phi = DynamicalSystem(F_list, domain=P)
        base_locus = phi.indeterminacy_locus()
    except Exception:
        base_locus = P.subscheme(F_list)
    return base_locus

# ==========================================
# Test Case 
# ==========================================
if __name__ == "__main__":
    k = QQ
    
    # Construct R = k[x]/(x^3)
    P.<x> = PolynomialRing(k)
    R = P.quotient(x^3, names='xbar')
    xbar = R.gen()
    
    v_list = [xbar, xbar^2]
    f, F = compute_exp_F(R, v_list)
    
    print(f"f = {f}")
    print(f"F = {F}")
    print(compute_base_locus(F))

f = (x1, 1/2*x1^2 + x2)
F = [x0^2, x0*x1, 1/2*x1^2 + x0*x2]
Closed subscheme of Projective Space of dimension 2 over Rational Field defined by:
  x1,
  x0


BASIS of m:

In [143]:
# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
R.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = R.ideal(x*y, z^2, x*z - y*z, x^2 + y^2 - x*z)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_R = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_R if b != R(1)]

# Output the results
print(f"Dimension of R: {len(basis_R)}")
print(f"Basis of R: {basis_R}")
print("-" * 30)
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")

Dimension of R: 6
Basis of R: [y*z, z, y^2, y, x, 1]
------------------------------
Dimension of m: 5
Basis of m: [y*z, z, y^2, y, x]


$\mathbb{P}^3$:

In [144]:
P.<x> = PolynomialRing(k)
R = P.quotient(x^4, names='xbar')
xbar = R.gen()
    
v_list = [xbar, xbar^2, xbar^3]
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(f"F = {F}")
print(compute_base_locus(F))

f = (x1, 1/2*x1^2 + x2, 1/6*x1^3 + x1*x2 + x3)
F = [x0^3, x0^2*x1, 1/2*x0*x1^2 + x0^2*x2, 1/6*x1^3 + x0*x1*x2 + x0^2*x3]
Closed subscheme of Projective Space of dimension 3 over Rational Field defined by:
  x1,
  x0


In [145]:
# ==========================================
# Test Case for k[x,y]/(x^2, xy, y^3)
# ==========================================
if __name__ == "__main__":
    k = QQ
    
    # Construct the multivariate polynomial ring
    P.<x, y> = PolynomialRing(k)
    
    # Define the ideal and the quotient ring R
    I = P.ideal(x^2, x*y, y^3)
    R = P.quotient(I, names=('xbar', 'ybar'))
    xbar, ybar = R.gens()
    
    # The maximal ideal basis: {x, y, y^2}
    v_list = [xbar, ybar, ybar^2]
    
    # Compute f and F
    f, F = compute_exp_F(R, v_list)
    
    print(f"f = {f}")
    print(latex(f))
    print(f"F = {F}")
    print(latex(F))

print(compute_base_locus(F))

f = (x1, x2, 1/2*x2^2 + x3)
\left(x_{1}, x_{2}, \frac{1}{2} x_{2}^{2} + x_{3}\right)
F = [x0^2, x0*x1, x0*x2, 1/2*x2^2 + x0*x3]
\left[x_{0}^{2}, x_{0} x_{1}, x_{0} x_{2}, \frac{1}{2} x_{2}^{2} + x_{0} x_{3}\right]
Closed subscheme of Projective Space of dimension 3 over Rational Field defined by:
  x2,
  x0


In [146]:
# Construct the multivariate polynomial ring
P.<x, y> = PolynomialRing(k)
    
# Define the ideal and the quotient ring R
I = P.ideal(x^2,y^2)
R = P.quotient(I, names=('xbar', 'ybar'))
xbar, ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, xbar*ybar]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {F}")
print(latex(F))
print(' ')
print(compute_base_locus(F))

f = (x1, x2, x1*x2 + x3)
\left(x_{1}, x_{2}, x_{1} x_{2} + x_{3}\right)
 
F = [x0^2, x0*x1, x0*x2, x1*x2 + x0*x3]
\left[x_{0}^{2}, x_{0} x_{1}, x_{0} x_{2}, x_{1} x_{2} + x_{0} x_{3}\right]
 
Closed subscheme of Projective Space of dimension 3 over Rational Field defined by:
  x0,
  x1*x2


$\mathbb{P}^4$:

In [147]:
# 4,1 k[x]/(x 5 )
P.<x> = PolynomialRing(k)
R = P.quotient(x^5, names='xbar')
xbar = R.gen()
    
v_list = [xbar, xbar^2, xbar^3, xbar^4]
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(f"F = {latex(F).replace(',', ':')}")
print(compute_base_locus(F))

f = (x1, 1/2*x1^2 + x2, 1/6*x1^3 + x1*x2 + x3, 1/24*x1^4 + 1/2*x1^2*x2 + 1/2*x2^2 + x1*x3 + x4)
F = \left[x_{0}^{4}: x_{0}^{3} x_{1}: \frac{1}{2} x_{0}^{2} x_{1}^{2} + x_{0}^{3} x_{2}: \frac{1}{6} x_{0} x_{1}^{3} + x_{0}^{2} x_{1} x_{2} + x_{0}^{3} x_{3}: \frac{1}{24} x_{1}^{4} + \frac{1}{2} x_{0} x_{1}^{2} x_{2} + \frac{1}{2} x_{0}^{2} x_{2}^{2} + x_{0}^{2} x_{1} x_{3} + x_{0}^{3} x_{4}\right]
Closed subscheme of Projective Space of dimension 4 over Rational Field defined by:
  x1,
  x0


In [148]:
# 4,2 k[x, y]/(x 2 , xy, y 4 ). [y^3, y^2, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x,y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,x*y, y^4)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, ybar^2, ybar^3]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 4
Basis of m: [y^3, y^2, y, x]
 
f = (x1, x2, 1/2*x2^2 + x3, 1/6*x2^3 + x2*x3 + x4)
\left(x_{1}, x_{2}, \frac{1}{2} x_{2}^{2} + x_{3}, \frac{1}{6} x_{2}^{3} + x_{2} x_{3} + x_{4}\right)
 
F = \left[x_{0}^{3}: x_{0}^{2} x_{1}: x_{0}^{2} x_{2}: \frac{1}{2} x_{0} x_{2}^{2} + x_{0}^{2} x_{3}: \frac{1}{6} x_{2}^{3} + x_{0} x_{2} x_{3} + x_{0}^{2} x_{4}\right]
 
Closed subscheme of Projective Space of dimension 4 over Rational Field defined by:
  x2,
  x0


In [149]:
# 4,3 k[x, y]/(x 2 + y 3 , xy). [y^2, y, x^2, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x,y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2+y^3,x*y)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, xbar^2, ybar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 4
Basis of m: [y^2, y, x^2, x]
 
f = (x1, x2, -1/6*x2^3 + 1/2*x1^2 - x2*x4 + x3, 1/2*x2^2 + x4)
\left(x_{1}, x_{2}, -\frac{1}{6} x_{2}^{3} + \frac{1}{2} x_{1}^{2} - x_{2} x_{4} + x_{3}, \frac{1}{2} x_{2}^{2} + x_{4}\right)
 
F = \left[x_{0}^{3}: x_{0}^{2} x_{1}: x_{0}^{2} x_{2}: \frac{1}{2} x_{0} x_{1}^{2} - \frac{1}{6} x_{2}^{3} + x_{0}^{2} x_{3} - x_{0} x_{2} x_{4}: \frac{1}{2} x_{0} x_{2}^{2} + x_{0}^{2} x_{4}\right]
 
Closed subscheme of Projective Space of dimension 4 over Rational Field defined by:
  x2,
  x0


In [150]:
# 4,4 k[x, y]/(xy, x 3 , y 3 ). [y^2, y, x^2, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x,y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x*y,x^3,y^3)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, xbar^2, ybar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 4
Basis of m: [y^2, y, x^2, x]
 
f = (x1, x2, 1/2*x1^2 + x3, 1/2*x2^2 + x4)
\left(x_{1}, x_{2}, \frac{1}{2} x_{1}^{2} + x_{3}, \frac{1}{2} x_{2}^{2} + x_{4}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: \frac{1}{2} x_{1}^{2} + x_{0} x_{3}: \frac{1}{2} x_{2}^{2} + x_{0} x_{4}\right]
 
Closed subscheme of Projective Space of dimension 4 over Rational Field defined by:
  x2,
  x1,
  x0


In [151]:
# 4,5 k[x, y]/(x 2 , xy 2 , y 3 ). [y^2, x*y, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x,y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,x*y^2,y^3)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, xbar*ybar, ybar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 4
Basis of m: [y^2, x*y, y, x]
 
f = (x1, x2, x1*x2 + x3, 1/2*x2^2 + x4)
\left(x_{1}, x_{2}, x_{1} x_{2} + x_{3}, \frac{1}{2} x_{2}^{2} + x_{4}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{1} x_{2} + x_{0} x_{3}: \frac{1}{2} x_{2}^{2} + x_{0} x_{4}\right]
 
Closed subscheme of Projective Space of dimension 4 over Rational Field defined by:
  x2,
  x0


In [152]:
# 4,6. k[x, y, z]/(x 2 , y 2 , xy, xz, yz, z 3 ). [z^2, z, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x,y,z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,y^2,x*y,x*z,y*z,z^3)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, zbar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 4
Basis of m: [z^2, z, y, x]
 
f = (x1, x2, x3, 1/2*x3^2 + x4)
\left(x_{1}, x_{2}, x_{3}, \frac{1}{2} x_{3}^{2} + x_{4}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: \frac{1}{2} x_{3}^{2} + x_{0} x_{4}\right]
 
Closed subscheme of Projective Space of dimension 4 over Rational Field defined by:
  x3,
  x0


In [153]:
# 4,7. k[x, y, z]/(x 2 , y 2 , z 2 , xy, xz). [y*z, z, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x,y,z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,y^2,z^2,x*y,x*z)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, ybar*zbar]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 4
Basis of m: [y*z, z, y, x]
 
f = (x1, x2, x3, x2*x3 + x4)
\left(x_{1}, x_{2}, x_{3}, x_{2} x_{3} + x_{4}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: x_{2} x_{3} + x_{0} x_{4}\right]
 
Closed subscheme of Projective Space of dimension 4 over Rational Field defined by:
  x0,
  x2*x3


In [154]:
# 4,8. k[x, y, z]/(xy, xz, yz, x 2 + y 2 , x 2 + z 2 ). [z^2, z, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x,y,z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x*y,x*z,y*z,x^2+y^2,x^2+z^2)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, zbar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 4
Basis of m: [z^2, z, y, x]
 
f = (x1, x2, x3, -1/2*x1^2 + 1/2*x2^2 + 1/2*x3^2 + x4)
\left(x_{1}, x_{2}, x_{3}, -\frac{1}{2} x_{1}^{2} + \frac{1}{2} x_{2}^{2} + \frac{1}{2} x_{3}^{2} + x_{4}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: -\frac{1}{2} x_{1}^{2} + \frac{1}{2} x_{2}^{2} + \frac{1}{2} x_{3}^{2} + x_{0} x_{4}\right]
 
Closed subscheme of Projective Space of dimension 4 over Rational Field defined by:
  x0,
  x1^2 - x2^2 - x3^2


$\mathbb{P}^5$:

In [155]:
# 5,1
P.<x> = PolynomialRing(k)
    
# Define the ideal and the quotient ring R
I = P.ideal(x^6)
R = P.quotient(I, names=('xbar'))
xbar = R.gen()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, xbar^2, xbar^3, xbar^4, xbar^5]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

f = (x1, 1/2*x1^2 + x2, 1/6*x1^3 + x1*x2 + x3, 1/24*x1^4 + 1/2*x1^2*x2 + 1/2*x2^2 + x1*x3 + x4, 1/120*x1^5 + 1/6*x1^3*x2 + 1/2*x1*x2^2 + 1/2*x1^2*x3 + x2*x3 + x1*x4 + x5)
\left(x_{1}, \frac{1}{2} x_{1}^{2} + x_{2}, \frac{1}{6} x_{1}^{3} + x_{1} x_{2} + x_{3}, \frac{1}{24} x_{1}^{4} + \frac{1}{2} x_{1}^{2} x_{2} + \frac{1}{2} x_{2}^{2} + x_{1} x_{3} + x_{4}, \frac{1}{120} x_{1}^{5} + \frac{1}{6} x_{1}^{3} x_{2} + \frac{1}{2} x_{1} x_{2}^{2} + \frac{1}{2} x_{1}^{2} x_{3} + x_{2} x_{3} + x_{1} x_{4} + x_{5}\right)
 
F = \left[x_{0}^{5}: x_{0}^{4} x_{1}: \frac{1}{2} x_{0}^{3} x_{1}^{2} + x_{0}^{4} x_{2}: \frac{1}{6} x_{0}^{2} x_{1}^{3} + x_{0}^{3} x_{1} x_{2} + x_{0}^{4} x_{3}: \frac{1}{24} x_{0} x_{1}^{4} + \frac{1}{2} x_{0}^{2} x_{1}^{2} x_{2} + \frac{1}{2} x_{0}^{3} x_{2}^{2} + x_{0}^{3} x_{1} x_{3} + x_{0}^{4} x_{4}: \frac{1}{120} x_{1}^{5} + \frac{1}{6} x_{0} x_{1}^{3} x_{2} + \frac{1}{2} x_{0}^{2} x_{1} x_{2}^{2} + \frac{1}{2} x_{0}^{2} x_{1}^{2} x_{3} + x_{0}^{3} x_{2} x_{3} + x_{0}

In [156]:
# 5,2 k[x, y]/(x 2 , xy, y 5 ). [y^4, y^3, y^2, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2, x*y,y^5)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, ybar^2, ybar^3, ybar^4]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [y^4, y^3, y^2, y, x]
 
f = (x1, x2, 1/2*x2^2 + x3, 1/6*x2^3 + x2*x3 + x4, 1/24*x2^4 + 1/2*x2^2*x3 + 1/2*x3^2 + x2*x4 + x5)
\left(x_{1}, x_{2}, \frac{1}{2} x_{2}^{2} + x_{3}, \frac{1}{6} x_{2}^{3} + x_{2} x_{3} + x_{4}, \frac{1}{24} x_{2}^{4} + \frac{1}{2} x_{2}^{2} x_{3} + \frac{1}{2} x_{3}^{2} + x_{2} x_{4} + x_{5}\right)
 
F = \left[x_{0}^{4}: x_{0}^{3} x_{1}: x_{0}^{3} x_{2}: \frac{1}{2} x_{0}^{2} x_{2}^{2} + x_{0}^{3} x_{3}: \frac{1}{6} x_{0} x_{2}^{3} + x_{0}^{2} x_{2} x_{3} + x_{0}^{3} x_{4}: \frac{1}{24} x_{2}^{4} + \frac{1}{2} x_{0} x_{2}^{2} x_{3} + \frac{1}{2} x_{0}^{2} x_{3}^{2} + x_{0}^{2} x_{2} x_{4} + x_{0}^{3} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x2,
  x0


In [157]:
# 5,3 k[x, y]/(x 2 + y 4 , xy). [y^3, y^2, y, x^2, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2+y^4, x*y)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, xbar^2, ybar^2, ybar^3]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [y^3, y^2, y, x^2, x]
 
f = (x1, x2, -1/24*x2^4 - 1/2*x2^2*x4 + 1/2*x1^2 - 1/2*x4^2 - x2*x5 + x3, 1/2*x2^2 + x4, 1/6*x2^3 + x2*x4 + x5)
\left(x_{1}, x_{2}, -\frac{1}{24} x_{2}^{4} - \frac{1}{2} x_{2}^{2} x_{4} + \frac{1}{2} x_{1}^{2} - \frac{1}{2} x_{4}^{2} - x_{2} x_{5} + x_{3}, \frac{1}{2} x_{2}^{2} + x_{4}, \frac{1}{6} x_{2}^{3} + x_{2} x_{4} + x_{5}\right)
 
F = \left[x_{0}^{4}: x_{0}^{3} x_{1}: x_{0}^{3} x_{2}: \frac{1}{2} x_{0}^{2} x_{1}^{2} - \frac{1}{24} x_{2}^{4} + x_{0}^{3} x_{3} - \frac{1}{2} x_{0} x_{2}^{2} x_{4} - \frac{1}{2} x_{0}^{2} x_{4}^{2} - x_{0}^{2} x_{2} x_{5}: \frac{1}{2} x_{0}^{2} x_{2}^{2} + x_{0}^{3} x_{4}: \frac{1}{6} x_{0} x_{2}^{3} + x_{0}^{2} x_{2} x_{4} + x_{0}^{3} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x2,
  x0


In [158]:
# 5,4 k[x, y]/(xy, x 3 , y 4 ). y^3, y^2, y, x^2, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x*y,x^3,y^4)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, xbar^2, ybar^2, ybar^3]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [y^3, y^2, y, x^2, x]
 
f = (x1, x2, 1/2*x1^2 + x3, 1/2*x2^2 + x4, 1/6*x2^3 + x2*x4 + x5)
\left(x_{1}, x_{2}, \frac{1}{2} x_{1}^{2} + x_{3}, \frac{1}{2} x_{2}^{2} + x_{4}, \frac{1}{6} x_{2}^{3} + x_{2} x_{4} + x_{5}\right)
 
F = \left[x_{0}^{3}: x_{0}^{2} x_{1}: x_{0}^{2} x_{2}: \frac{1}{2} x_{0} x_{1}^{2} + x_{0}^{2} x_{3}: \frac{1}{2} x_{0} x_{2}^{2} + x_{0}^{2} x_{4}: \frac{1}{6} x_{2}^{3} + x_{0} x_{2} x_{4} + x_{0}^{2} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x2,
  x0


In [159]:
# 5,5 k[x, y]/(xy, x 3 + y 3 ). [y^3, y^2, y, x^2, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x*y,x^3+y^3)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, xbar^2, ybar^2, ybar^3]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [y^3, y^2, y, x^2, x]
 
f = (x1, x2, 1/2*x1^2 + x3, 1/2*x2^2 + x4, -1/6*x1^3 + 1/6*x2^3 - x1*x3 + x2*x4 + x5)
\left(x_{1}, x_{2}, \frac{1}{2} x_{1}^{2} + x_{3}, \frac{1}{2} x_{2}^{2} + x_{4}, -\frac{1}{6} x_{1}^{3} + \frac{1}{6} x_{2}^{3} - x_{1} x_{3} + x_{2} x_{4} + x_{5}\right)
 
F = \left[x_{0}^{3}: x_{0}^{2} x_{1}: x_{0}^{2} x_{2}: \frac{1}{2} x_{0} x_{1}^{2} + x_{0}^{2} x_{3}: \frac{1}{2} x_{0} x_{2}^{2} + x_{0}^{2} x_{4}: -\frac{1}{6} x_{1}^{3} + \frac{1}{6} x_{2}^{3} - x_{0} x_{1} x_{3} + x_{0} x_{2} x_{4} + x_{0}^{2} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x0,
  x1^3 - x2^3


In [160]:
# 5,6 k[x, y]/(x 2 , xy 2 , y 4 ). [y^3, y^2, x*y, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,x*y^2,y^4)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, xbar*ybar, ybar^2, ybar^3]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [y^3, y^2, x*y, y, x]
 
f = (x1, x2, x1*x2 + x3, 1/2*x2^2 + x4, 1/6*x2^3 + x2*x4 + x5)
\left(x_{1}, x_{2}, x_{1} x_{2} + x_{3}, \frac{1}{2} x_{2}^{2} + x_{4}, \frac{1}{6} x_{2}^{3} + x_{2} x_{4} + x_{5}\right)
 
F = \left[x_{0}^{3}: x_{0}^{2} x_{1}: x_{0}^{2} x_{2}: x_{0} x_{1} x_{2} + x_{0}^{2} x_{3}: \frac{1}{2} x_{0} x_{2}^{2} + x_{0}^{2} x_{4}: \frac{1}{6} x_{2}^{3} + x_{0} x_{2} x_{4} + x_{0}^{2} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x2,
  x0


In [161]:
# 5,7 k[x, y]/(x 2 + y 3 , xy 2 , y 4 ) [y^2, x*y, y, x^2, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2+y^3,x*y^2,y^4)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, xbar*ybar, xbar^2, ybar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [y^2, x*y, y, x^2, x]
 
f = (x1, x2, x1*x2 + x3, -1/6*x2^3 + 1/2*x1^2 - x2*x5 + x4, 1/2*x2^2 + x5)
\left(x_{1}, x_{2}, x_{1} x_{2} + x_{3}, -\frac{1}{6} x_{2}^{3} + \frac{1}{2} x_{1}^{2} - x_{2} x_{5} + x_{4}, \frac{1}{2} x_{2}^{2} + x_{5}\right)
 
F = \left[x_{0}^{3}: x_{0}^{2} x_{1}: x_{0}^{2} x_{2}: x_{0} x_{1} x_{2} + x_{0}^{2} x_{3}: \frac{1}{2} x_{0} x_{1}^{2} - \frac{1}{6} x_{2}^{3} + x_{0}^{2} x_{4} - x_{0} x_{2} x_{5}: \frac{1}{2} x_{0} x_{2}^{2} + x_{0}^{2} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x2,
  x0


In [162]:
# 5,8 k[x, y]/(x 2 , y 3 ) [x*y^2, y^2, x*y, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,y^3)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, xbar*ybar, ybar^2, xbar*ybar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [x*y^2, y^2, x*y, y, x]
 
f = (x1, x2, x1*x2 + x3, 1/2*x2^2 + x4, 1/2*x1*x2^2 + x2*x3 + x1*x4 + x5)
\left(x_{1}, x_{2}, x_{1} x_{2} + x_{3}, \frac{1}{2} x_{2}^{2} + x_{4}, \frac{1}{2} x_{1} x_{2}^{2} + x_{2} x_{3} + x_{1} x_{4} + x_{5}\right)
 
F = \left[x_{0}^{3}: x_{0}^{2} x_{1}: x_{0}^{2} x_{2}: x_{0} x_{1} x_{2} + x_{0}^{2} x_{3}: \frac{1}{2} x_{0} x_{2}^{2} + x_{0}^{2} x_{4}: \frac{1}{2} x_{1} x_{2}^{2} + x_{0} x_{2} x_{3} + x_{0} x_{1} x_{4} + x_{0}^{2} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x0,
  x1*x2


In [163]:
# 5,9 k[x, y]/(x, y)3 [y^2, x*y, y, x^2, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^3,x^2*y,x*y^2,y^3)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar'))
xbar,ybar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, xbar*ybar, xbar^2, ybar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [y^2, x*y, y, x^2, x]
 
f = (x1, x2, x1*x2 + x3, 1/2*x1^2 + x4, 1/2*x2^2 + x5)
\left(x_{1}, x_{2}, x_{1} x_{2} + x_{3}, \frac{1}{2} x_{1}^{2} + x_{4}, \frac{1}{2} x_{2}^{2} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{1} x_{2} + x_{0} x_{3}: \frac{1}{2} x_{1}^{2} + x_{0} x_{4}: \frac{1}{2} x_{2}^{2} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x2,
  x1,
  x0


In [164]:
# 5,10 k[x, y, z]/(x 2 , xy, y 2 , xz, yz, z 4 ) [z^3, z^2, z, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,x*y,y^2,x*z,y*z,z^4)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, zbar^2, zbar^3]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [z^3, z^2, z, y, x]
 
f = (x1, x2, x3, 1/2*x3^2 + x4, 1/6*x3^3 + x3*x4 + x5)
\left(x_{1}, x_{2}, x_{3}, \frac{1}{2} x_{3}^{2} + x_{4}, \frac{1}{6} x_{3}^{3} + x_{3} x_{4} + x_{5}\right)
 
F = \left[x_{0}^{3}: x_{0}^{2} x_{1}: x_{0}^{2} x_{2}: x_{0}^{2} x_{3}: \frac{1}{2} x_{0} x_{3}^{2} + x_{0}^{2} x_{4}: \frac{1}{6} x_{3}^{3} + x_{0} x_{3} x_{4} + x_{0}^{2} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x3,
  x0


In [165]:
# 5,11 k[x, y, z]/(x 2 , xy, y 2 + z 3 , xz, yz) [z^2, z, y^2, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,x*y,y^2+z^3,x*z,y*z)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, ybar^2, zbar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [z^2, z, y^2, y, x]
 
f = (x1, x2, x3, -1/6*x3^3 + 1/2*x2^2 - x3*x5 + x4, 1/2*x3^2 + x5)
\left(x_{1}, x_{2}, x_{3}, -\frac{1}{6} x_{3}^{3} + \frac{1}{2} x_{2}^{2} - x_{3} x_{5} + x_{4}, \frac{1}{2} x_{3}^{2} + x_{5}\right)
 
F = \left[x_{0}^{3}: x_{0}^{2} x_{1}: x_{0}^{2} x_{2}: x_{0}^{2} x_{3}: \frac{1}{2} x_{0} x_{2}^{2} - \frac{1}{6} x_{3}^{3} + x_{0}^{2} x_{4} - x_{0} x_{3} x_{5}: \frac{1}{2} x_{0} x_{3}^{2} + x_{0}^{2} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x3,
  x0


In [166]:
# 5,12 k[x, y, z]/(x 2 , xy + z 3 , y 2 , xz, yz) [z^2, z, x*y, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,x*y+z^3,y^2,x*z,y*z)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, xbar*ybar, zbar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [z^2, z, x*y, y, x]
 
f = (x1, x2, x3, -1/6*x3^3 + x1*x2 - x3*x5 + x4, 1/2*x3^2 + x5)
\left(x_{1}, x_{2}, x_{3}, -\frac{1}{6} x_{3}^{3} + x_{1} x_{2} - x_{3} x_{5} + x_{4}, \frac{1}{2} x_{3}^{2} + x_{5}\right)
 
F = \left[x_{0}^{3}: x_{0}^{2} x_{1}: x_{0}^{2} x_{2}: x_{0}^{2} x_{3}: x_{0} x_{1} x_{2} - \frac{1}{6} x_{3}^{3} + x_{0}^{2} x_{4} - x_{0} x_{3} x_{5}: \frac{1}{2} x_{0} x_{3}^{2} + x_{0}^{2} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x3,
  x0


In [167]:
# 5,13 k[x, y, z]/(xy, yz, z 2 , y 2 − xz, x 3 ) [x*z, z, y, x^2, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x*y,y*z,z^2,y^2-x*z,x^3)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, xbar^2, xbar*zbar]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [x*z, z, y, x^2, x]
 
f = (x1, x2, x3, 1/2*x1^2 + x4, 1/2*x2^2 + x1*x3 + x5)
\left(x_{1}, x_{2}, x_{3}, \frac{1}{2} x_{1}^{2} + x_{4}, \frac{1}{2} x_{2}^{2} + x_{1} x_{3} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: \frac{1}{2} x_{1}^{2} + x_{0} x_{4}: \frac{1}{2} x_{2}^{2} + x_{1} x_{3} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x2,
  x1,
  x0


In [168]:
# 5,14 k[x,y,z]/(xy, z^2, xz-yz, x^2+y^2-xz). Basis of m: [y*z, z, y^2, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x*y, z^2, x*z - y*z, x^2 + y^2 - x*z)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, ybar^2, ybar*zbar]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [y*z, z, y^2, y, x]
 
f = (x1, x2, x3, -1/2*x1^2 + 1/2*x2^2 + x4, 1/2*x1^2 + x1*x3 + x2*x3 + x5)
\left(x_{1}, x_{2}, x_{3}, -\frac{1}{2} x_{1}^{2} + \frac{1}{2} x_{2}^{2} + x_{4}, \frac{1}{2} x_{1}^{2} + x_{1} x_{3} + x_{2} x_{3} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: -\frac{1}{2} x_{1}^{2} + \frac{1}{2} x_{2}^{2} + x_{0} x_{4}: \frac{1}{2} x_{1}^{2} + x_{1} x_{3} + x_{2} x_{3} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x1 - x2,
  x0,
  x2^2 + 4*x2*x3


In [169]:
# 5,15 k[x, y, z]/(x 2 , xy, xz, y 2 , yz 2 , z 3 ) [z^2, y*z, z, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,x*y,x*z,y^2,y*z^2,z^3)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, ybar*zbar, zbar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [z^2, y*z, z, y, x]
 
f = (x1, x2, x3, x2*x3 + x4, 1/2*x3^2 + x5)
\left(x_{1}, x_{2}, x_{3}, x_{2} x_{3} + x_{4}, \frac{1}{2} x_{3}^{2} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: x_{2} x_{3} + x_{0} x_{4}: \frac{1}{2} x_{3}^{2} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x3,
  x0


In [170]:
# 5,16 k[x, y, z]/(x 2 , xy, xz, yz, y 3 , z 3 ) [z^2, z, y^2, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,x*y,x*z,y*z,y^3,z^3)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, ybar^2, zbar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [z^2, z, y^2, y, x]
 
f = (x1, x2, x3, 1/2*x2^2 + x4, 1/2*x3^2 + x5)
\left(x_{1}, x_{2}, x_{3}, \frac{1}{2} x_{2}^{2} + x_{4}, \frac{1}{2} x_{3}^{2} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: \frac{1}{2} x_{2}^{2} + x_{0} x_{4}: \frac{1}{2} x_{3}^{2} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x3,
  x2,
  x0


In [171]:
# 5,17 k[x, y, z]/(xy, xz, y 2 , z 2 , x 3 ) [y*z, z, y, x^2, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x*y,x*z,y^2,z^2,x^3)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, xbar^2, ybar*zbar]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [y*z, z, y, x^2, x]
 
f = (x1, x2, x3, 1/2*x1^2 + x4, x2*x3 + x5)
\left(x_{1}, x_{2}, x_{3}, \frac{1}{2} x_{1}^{2} + x_{4}, x_{2} x_{3} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: \frac{1}{2} x_{1}^{2} + x_{0} x_{4}: x_{2} x_{3} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x1,
  x0,
  x2*x3


In [172]:
# 5,18 k[x, y, z]/(xy, xz, yz, x 2 + y 2 − z 2 ) [z^2, z, y^2, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x*y,x*z,y*z,x^2+y^2-z^2)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, ybar^2, zbar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [z^2, z, y^2, y, x]
 
f = (x1, x2, x3, -1/2*x1^2 + 1/2*x2^2 + x4, 1/2*x1^2 + 1/2*x3^2 + x5)
\left(x_{1}, x_{2}, x_{3}, -\frac{1}{2} x_{1}^{2} + \frac{1}{2} x_{2}^{2} + x_{4}, \frac{1}{2} x_{1}^{2} + \frac{1}{2} x_{3}^{2} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: -\frac{1}{2} x_{1}^{2} + \frac{1}{2} x_{2}^{2} + x_{0} x_{4}: \frac{1}{2} x_{1}^{2} + \frac{1}{2} x_{3}^{2} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x0,
  x2^2 + x3^2,
  x1^2 + x3^2


In [173]:
# 5,19 k[x, y, z]/(x 2 , xy, yz, y 2 − z 2 ) [z^2, x*z, z, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,x*y,y*z,y^2-z^2)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, xbar*zbar, zbar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [z^2, x*z, z, y, x]
 
f = (x1, x2, x3, x1*x3 + x4, 1/2*x2^2 + 1/2*x3^2 + x5)
\left(x_{1}, x_{2}, x_{3}, x_{1} x_{3} + x_{4}, \frac{1}{2} x_{2}^{2} + \frac{1}{2} x_{3}^{2} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: x_{1} x_{3} + x_{0} x_{4}: \frac{1}{2} x_{2}^{2} + \frac{1}{2} x_{3}^{2} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x0,
  x1*x3,
  x2^2 + x3^2,
  x1*x2


In [174]:
# 5,20 k[x, y, z]/(x 2 , xy, y 2 , z 2 ) [y*z, x*z, z, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,x*y,y^2,z^2)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar'))
xbar,ybar,zbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, xbar*zbar, ybar*zbar]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [y*z, x*z, z, y, x]
 
f = (x1, x2, x3, x1*x3 + x4, x2*x3 + x5)
\left(x_{1}, x_{2}, x_{3}, x_{1} x_{3} + x_{4}, x_{2} x_{3} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: x_{1} x_{3} + x_{0} x_{4}: x_{2} x_{3} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x0,
  x2*x3,
  x1*x3


In [175]:
# 5,21 k[x, y, z, w]/(x 2 , y 2 , z 2 , xy, xz, xw, yz, yw, zw, w 3 ) [w^2, w, z, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z, w> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,y^2,z^2,x*y,x*z,x*w,y*z,y*w,z*w,w^3)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar','wbar'))
xbar,ybar,zbar,wbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, wbar, wbar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [w^2, w, z, y, x]
 
f = (x1, x2, x3, x4, 1/2*x4^2 + x5)
\left(x_{1}, x_{2}, x_{3}, x_{4}, \frac{1}{2} x_{4}^{2} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: x_{0} x_{4}: \frac{1}{2} x_{4}^{2} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x4,
  x0


In [176]:
# 5,22 k[x, y, z, w]/(x 2 , y 2 , z 2 , w 2 , xy, xz, xw, yz, yw) [z*w, w, z, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z, w> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,y^2,z^2,w^2,x*y,x*z,x*w,y*z,y*w)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar','wbar'))
xbar,ybar,zbar,wbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, wbar, zbar*wbar]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [z*w, w, z, y, x]
 
f = (x1, x2, x3, x4, x3*x4 + x5)
\left(x_{1}, x_{2}, x_{3}, x_{4}, x_{3} x_{4} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: x_{0} x_{4}: x_{3} x_{4} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x0,
  x3*x4


In [177]:
# 5,23 k[x, y, z, w]/(x 2 , y 2 + z 2 , y 2 + w 2 , xy, xz, xw, yz, yw, zw) [w^2, w, z, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z, w> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,y^2+z^2,y^2+w^2,x*y,x*z,x*w,y*z,y*w,z*w)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar','wbar'))
xbar,ybar,zbar,wbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, wbar, wbar^2]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [w^2, w, z, y, x]
 
f = (x1, x2, x3, x4, -1/2*x2^2 + 1/2*x3^2 + 1/2*x4^2 + x5)
\left(x_{1}, x_{2}, x_{3}, x_{4}, -\frac{1}{2} x_{2}^{2} + \frac{1}{2} x_{3}^{2} + \frac{1}{2} x_{4}^{2} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: x_{0} x_{4}: -\frac{1}{2} x_{2}^{2} + \frac{1}{2} x_{3}^{2} + \frac{1}{2} x_{4}^{2} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x0,
  x2^2 - x3^2 - x4^2


In [178]:
# 5,24 k[x, y, z, w]/(x 2 , y 2 , z 2 , w 2 , xy − zw, xz, xw, yz, yw) [z*w, w, z, y, x]

# 1. Define the polynomial ring over a field of characteristic 0 (e.g., QQ)
P.<x, y, z, w> = PolynomialRing(QQ, order='degrevlex')

# 2. Define the defining ideal I
I = P.ideal(x^2,y^2,z^2,w^2,x*y-z*w,x*z,x*w,y*z,y*w)

# 3. Compute the k-basis of the quotient ring R using the ideal's normal_basis() method
basis_P = I.normal_basis()

# 4. The basis of the maximal ideal m is the basis of R excluding the constant 1.
# Note: we compare to R(1) since the elements of normal_basis() live in the polynomial ring R.
basis_m = [b for b in basis_P if b != R(1)]

# Output the results
print(f"Dimension of m: {len(basis_m)}")
print(f"Basis of m: {basis_m}")
print(' ')

R = P.quotient(I, names=('xbar','ybar','zbar','wbar'))
xbar,ybar,zbar,wbar = R.gens()
    
# The maximal ideal basis: {x, y, y^2}
v_list = [xbar, ybar, zbar, wbar, zbar*wbar]
    
# Compute f and F
f, F = compute_exp_F(R, v_list)
    
print(f"f = {f}")
print(latex(f))
print(' ')
print(f"F = {latex(F).replace(',', ':')}")
print(' ')
print(compute_base_locus(F))

Dimension of m: 5
Basis of m: [z*w, w, z, y, x]
 
f = (x1, x2, x3, x4, x1*x2 + x3*x4 + x5)
\left(x_{1}, x_{2}, x_{3}, x_{4}, x_{1} x_{2} + x_{3} x_{4} + x_{5}\right)
 
F = \left[x_{0}^{2}: x_{0} x_{1}: x_{0} x_{2}: x_{0} x_{3}: x_{0} x_{4}: x_{1} x_{2} + x_{3} x_{4} + x_{0} x_{5}\right]
 
Closed subscheme of Projective Space of dimension 5 over Rational Field defined by:
  x0,
  x1*x2 + x3*x4
